# CED — identificação SysIdentPy e exportação para MATLAB

Abra este notebook no VS Code, selecione um kernel Python 3.12 e execute as células em ordem (Run All). A primeira célula instala as dependências no kernel selecionado. É necessária internet para a instalação e o primeiro download.

Este notebook segue a **versão online consultada em 9 de setembro de 2026**, não uma reprodução certificada de uma edição PDF anterior:
- [Livro, capítulo 10 — Coupled Electric Device](https://sysidentpy.org/book/10-Case-Studies/#coupled-eletric-device)
- [Tutorial correspondente](https://sysidentpy.org/user-guide/tutorials/coupled-eletric-device/)
- [Carregador oficial dos benchmarks](https://github.com/MaartenSchoukens/nonlinear_benchmarks)

São seis modelos independentes (três configurações × dois experimentos). O carregador fornece os dados de treino/teste: não aplicamos normalização adicional, decimação nem nova divisão. A janela inicial do teste é usada apenas para inicialização e excluída das métricas. A versão online atual usa no máximo 10 atrasos, diferentemente das antigas configurações com 14 atrasos.

Os resultados são criados na pasta `CED_outputs` do diretório de execução. O notebook não precisa do MATLAB para identificar ou exportar os modelos.

Validação local concluída: os seis modelos foram executados com os dados CED e as equações exportadas reproduziram a simulação do SysIdentPy dentro da tolerância numérica definida. As células permanecem sem saídas gravadas para facilitar sua execução no VS Code.


In [ ]:
%pip install sysidentpy==0.9.0 nonlinear-benchmarks==1.0.1 numpy scipy pandas matplotlib


In [ ]:
from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import platform
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import savemat
from IPython.display import display, Markdown
import nonlinear_benchmarks
from sysidentpy.model_structure_selection import FROLS
from sysidentpy.basis_function import Polynomial
from sysidentpy.parameter_estimation import LeastSquares
from sysidentpy.metrics import root_mean_squared_error

OUT = Path("CED_outputs").resolve()
OUT.mkdir(parents=True, exist_ok=True)
versions = {p: version(p) for p in ["sysidentpy", "nonlinear-benchmarks", "numpy", "scipy", "pandas", "matplotlib"]}
versions["python"] = platform.python_version()
print(json.dumps(versions, indent=2))
print("Pasta de saída:", OUT)


## 1. Baixar e preservar os dois experimentos

A URL abaixo é a mesma usada no exemplo online atual. Se a fonte estiver indisponível, a célula interrompe com o erro de download; não substitui silenciosamente o dataset. Os hashes registrados permitem conferir se os arrays usados em execuções futuras são idênticos.


In [ ]:
CED_URL = (
    "https://web.archive.org/web/20210117142533id_/"
    "http://www.it.uu.se/research/publications/reports/2010-020/NonlinearData.zip"
)
train_val, test = nonlinear_benchmarks.CED(url=CED_URL, atleast_2d=True)
assert len(train_val) == len(test) == 2, "Esperados dois experimentos."
experiments = {}
data_manifest = {}
for exp, (training, testing) in enumerate(zip(train_val, test), start=1):
    d = {
        "u_train": np.asarray(training.u, dtype=float).reshape(-1, 1),
        "y_train": np.asarray(training.y, dtype=float).reshape(-1, 1),
        "u_test": np.asarray(testing.u, dtype=float).reshape(-1, 1),
        "y_test": np.asarray(testing.y, dtype=float).reshape(-1, 1),
        "n_initial": int(testing.state_initialization_window_length),
        "Ts": float(training.sampling_time),
    }
    assert d["u_train"].shape == d["y_train"].shape
    assert d["u_test"].shape == d["y_test"].shape
    assert 0 < d["n_initial"] < len(d["y_test"])
    for key in ["u_train", "y_train", "u_test", "y_test"]:
        assert np.isfinite(d[key]).all(), f"Valores não finitos em {key}"
    experiments[exp] = d
    savemat(OUT / f"CED_experiment_{exp}_data.mat", d)
    data_manifest[str(exp)] = {
        "n_train": len(d["y_train"]), "n_test": len(d["y_test"]),
        "n_initial": d["n_initial"],
        "sha256": {k: hashlib.sha256(np.asarray(d[k], dtype="<f8").tobytes()).hexdigest()
                   for k in ["u_train", "y_train", "u_test", "y_test"]}
    }
    print(f"Experimento {exp}: treino={len(d['y_train'])}, teste={len(d['y_test'])}, inicialização={d['n_initial']}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7), constrained_layout=True)
for exp, d in experiments.items():
    for col, split in enumerate(["train", "test"]):
        ax = axes[exp - 1, col]
        ax.plot(d[f"u_{split}"], label="Entrada u", alpha=0.8)
        ax.plot(d[f"y_{split}"], label="Saída y", color="black")
        if split == "test":
            ax.axvspan(0, d["n_initial"] - 1, color="grey", alpha=0.2, label="Inicialização")
        ax.set(title=f"Experimento {exp} — {split}", xlabel="Índice da amostra (Python)")
        ax.grid(alpha=0.25)
        ax.legend()
plt.show()


## 2. Configurações do livro

| Identificador | Grau | Atrasos de u e y | Seleção |
|---|---:|---:|---|
| degree2_lag7_aic | 2 | 7 | AIC, 120 valores |
| degree2_lag10 | 2 | 10 | Até 22 termos, ERR acumulado 0.9996 |
| degree3_lag10 | 3 | 10 | Até 40 termos, ERR acumulado 0.9996 |

Todos usam `LeastSquares()` como na página atual. Não adicionamos termos de resíduos: os modelos são NARX. A previsão é simulação livre, sem realimentar a saída medida depois da inicialização.

O RMSE é a métrica apresentada no exemplo. Também exportamos **NRMSE_std = RMSE / std(y avaliado)**, com desvio padrão populacional, e **NRMSE_range = RMSE / (max(y) − min(y))**. Para comparar com seu Green, escolha exatamente a mesma definição e as mesmas amostras nos dois modelos.


In [ ]:
CONFIGS = [
    {"name": "degree2_lag7_aic", "degree": 2, "lag": 7},
    {"name": "degree2_lag10", "degree": 2, "lag": 10},
    {"name": "degree3_lag10", "degree": 3, "lag": 10},
]

def build_model(cfg, exp):
    kwargs = dict(xlag=cfg["lag"], ylag=cfg["lag"],
                  basis_function=Polynomial(degree=cfg["degree"]),
                  estimator=LeastSquares())
    if cfg["name"] == "degree2_lag7_aic":
        kwargs.update(info_criteria="aic", n_info_values=120)
    else:
        kwargs.update(err_tol=0.9996, n_terms=22 if cfg["degree"] == 2 else 40,
                      order_selection=False)
        if exp == 2:
            kwargs["info_criteria"] = "aicc"  # presente no código do livro
    return FROLS(**kwargs)

def regressor_names(codes):
    names = []
    for row in np.asarray(codes):
        factors = []
        for c in row:
            c = int(c)
            if c == 0:
                continue
            signal, lag = divmod(c, 1000)
            assert signal in (1, 2), f"Código inesperado para CED SISO: {c}"
            factors.append(f"{'y' if signal == 1 else 'u'}(k-{lag})")
        names.append(" * ".join(factors) if factors else "1")
    return names

def simulate_exported(codes, theta, u, y_initial):
    # Avaliação independente da equação exportada; mesma inicialização do predict.
    u = np.asarray(u).ravel()
    y_initial = np.asarray(y_initial).ravel()
    y = np.zeros(len(u), dtype=float)
    y[:len(y_initial)] = y_initial
    for k in range(len(y_initial), len(u)):
        total = 0.0
        for row, coefficient in zip(codes, np.asarray(theta).ravel()):
            term = 1.0
            for c in row:
                c = int(c)
                if c:
                    signal, lag = divmod(c, 1000)
                    term *= y[k-lag] if signal == 1 else u[k-lag]
            total += coefficient * term
        y[k] = total
    return y.reshape(-1, 1)


In [ ]:
models = {}
summary_rows = []
for cfg in CONFIGS:
    for exp, d in experiments.items():
        key = f"CED_exp{exp}_{cfg['name']}"
        print(f"Identificando {key}...", flush=True)
        model = build_model(cfg, exp)
        t0 = time.perf_counter()
        model.fit(X=d["u_train"], y=d["y_train"])
        fit_seconds = time.perf_counter() - t0
        lag = int(model.max_lag)
        n = d["n_initial"]
        if lag > n:
            raise ValueError("Atraso maior que a janela de inicialização do benchmark.")
        start = n - lag
        yhat_full = model.predict(X=d["u_test"][start:], y=d["y_test"][start:n])
        yhat = np.asarray(yhat_full)[lag:].reshape(-1, 1)
        target = d["y_test"][n:]
        assert yhat.shape == target.shape
        if not np.isfinite(yhat).all():
            raise RuntimeError(f"Simulação não finita: {key}")
        replay = simulate_exported(model.final_model, model.theta,
                                    d["u_test"][start:], d["y_test"][start:n])[lag:]
        np.testing.assert_allclose(replay, yhat, rtol=1e-6, atol=1e-8,
                                   err_msg="Equação exportada diverge do predict.")
        rmse = float(root_mean_squared_error(target, yhat))
        std = float(np.std(target))
        span = float(np.ptp(target))
        sst = float(np.sum((target - np.mean(target)) ** 2))
        metrics = {
            "RMSE": rmse,
            "NRMSE_std": rmse / std if std else float("nan"),
            "NRMSE_range": rmse / span if span else float("nan"),
            "R2": 1 - float(np.sum((target-yhat)**2)) / sst if sst else float("nan"),
        }
        codes = np.asarray(model.final_model, dtype=np.int64)
        theta = np.asarray(model.theta).reshape(-1, 1)
        labels = regressor_names(codes)
        table = pd.DataFrame({"regressor": labels, "coefficient": theta.ravel()})
        table.to_csv(OUT / f"{key}_terms.csv", index=False, float_format="%.17g")
        equation = "y(k) = " + " + ".join(f"({v:.17g}) * {r}" for r, v in zip(labels, theta.ravel()))
        (OUT / f"{key}_equation.txt").write_text(equation + "\n", encoding="utf-8")
        savemat(OUT / f"{key}_model.mat", {
            "theta": theta, "model_code": codes,
            "regressor_labels": np.asarray(labels, dtype=object).reshape(-1, 1),
            "max_lag": lag, "degree": cfg["degree"], "n_terms": len(theta),
            "experiment": exp, "n_initial": n,
            "evaluation_start_matlab": n + 1,
            "simulation_start_matlab": start + 1,
            "y_initial": d["y_test"][start:n],
            "y_test_evaluated": target, "yhat_test": yhat,
            **metrics,
        })
        models[key] = {"model": model, "yhat": yhat, "target": target,
                       "experiment": exp, "metrics": metrics, "table": table}
        summary_rows.append({"model": key, "experiment": exp, "configuration": cfg["name"],
                             "n_terms": len(theta), "max_lag": lag,
                             "fit_seconds": fit_seconds, **metrics})
        print(f"  {len(theta)} termos; RMSE={rmse:.6f}; equação exportada verificada.")
        display(table)

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT / "CED_summary.csv", index=False, float_format="%.17g")
display(summary)


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 12), constrained_layout=True)
for row, cfg in enumerate(CONFIGS):
    for col, exp in enumerate([1, 2]):
        key = f"CED_exp{exp}_{cfg['name']}"
        result = models[key]
        n = experiments[exp]["n_initial"]
        k = np.arange(n, n + len(result["target"]))
        ax = axes[row, col]
        ax.plot(k, result["target"].ravel(), color="black", linewidth=3.5, label="Data")
        ax.plot(k, result["yhat"].ravel(), color="#822327", linewidth=2,
                marker="o", markevery=5, markersize=8, markerfacecolor="none", label="SysIdentPy")
        ax.set(title=f"Exp. {exp} — {cfg['name']}\nRMSE={result['metrics']['RMSE']:.6f}",
               xlabel="Índice da amostra (Python)", ylabel="Saída y")
        ax.grid(alpha=0.25)
        ax.legend()
fig.savefig(OUT / "CED_free_run_comparison.png", dpi=180)
plt.show()

display(summary.groupby("configuration")[["RMSE", "NRMSE_std"]].mean().rename(
    columns={"RMSE": "mean_RMSE", "NRMSE_std": "mean_NRMSE_std"}))
manifest = {"source_url": CED_URL, "book_url": "https://sysidentpy.org/book/10-Case-Studies/",
            "versions": versions, "data": data_manifest, "configurations": CONFIGS,
            "evaluation": "Free run; exclude n_initial samples; no added normalization or decimation",
            "NRMSE_std_definition": "RMSE / std(y_test[n_initial:], ddof=0)"}
(OUT / "CED_run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Concluído. Arquivos guardados em:", OUT)


## 3. Usar no MATLAB

Exemplo para o primeiro experimento e a configuração de grau 2 / 7 atrasos:

```matlab
D = load('CED_outputs/CED_experiment_1_data.mat');
M = load('CED_outputs/CED_exp1_degree2_lag7_aic_model.mat');
u_id = D.u_train;
y_id = D.y_train;
u_val = D.u_test;
y_val = D.y_test;
reference.theta = M.theta;
reference.model_code = M.model_code;
```

`model_code` tem uma linha por coeficiente e uma coluna por fator do monômio:
- `0`: fator constante 1;
- `1000 + lag`: saída `y(k-lag)`;
- `2000 + lag`: entrada `u(k-lag)`.

Exemplo: `[1001, 2002]` representa `y(k-1)*u(k-2)`. Os códigos não são índices diretos de um vetor MATLAB: devem ser decodificados. O arquivo `_equation.txt` mostra a mesma equação por extenso, sem precisar interpretar os códigos manualmente.

`M.yhat_test` contém somente as amostras avaliadas, correspondentes a `D.y_test(M.evaluation_start_matlab:end)`. Inicialize a referência e o Green de modo equivalente, preserve a janela de avaliação e use a mesma definição de NRMSE. Todos os coeficientes são exportados como float64; não são copiados de tabelas arredondadas.

**Validação:** o notebook verifica, para cada modelo, se uma implementação independente da equação exportada reproduz `model.predict`. As saídas do notebook ficam vazias até você executá-lo; resultados só devem ser reportados depois dessa execução.
